# Grad-CAM — Visual Explanations from Deep Networks via Gradient-based Localization (2017)

_Papers · Meta-learning, Bayesian & Robustness_

**Turn the gradient of a class score into a heatmap showing which image region the network looked at.**

---

This notebook is a practice scaffold for the **Grad-CAM — Visual Explanations from Deep Networks via Gradient-based Localization (2017)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch, torch.nn as nn, numpy as np
torch.manual_seed(0); np.random.seed(0)

# --- 0. Sanity-check the worked 2x2 example: one feature map, Z=4. ---
A1 = torch.tensor([[2., 0.], [1., 4.]])          # the feature map A^1
dA = torch.tensor([[1., 3.], [-2., 0.]])         # dy^c / dA^1 (from backprop)
alpha = dA.mean()                                # Eqn 1: global-average-pool the gradient
cam = torch.relu(alpha * A1)                     # Eqn 2: ReLU(alpha * A), one map
print("worked example: alpha =", alpha.item(), " heatmap =", cam.tolist())
# worked example: alpha = 0.5  heatmap = [[1.0, 0.0], [0.5, 2.0]]


# --- 1. Toy task: 8x8 images. Class 0 = blob top-left, class 1 = blob bottom-right. ---
H = 8
def make_image(cls):
    img = np.random.uniform(0, 0.1, (H, H)).astype(np.float32)
    cy, cx = (np.random.randint(0,3), np.random.randint(0,3)) if cls == 0 \
             else (np.random.randint(5,8), np.random.randint(5,8))
    for dy in (-1,0,1):
        for dx in (-1,0,1):
            y, x = cy+dy, cx+dx
            if 0 <= y < H and 0 <= x < H: img[y, x] += 1.0
    return img
def batch(n):
    cs = np.random.randint(0, 2, n)
    X = torch.tensor(np.stack([make_image(c) for c in cs])).unsqueeze(1)  # (n,1,8,8)
    return X, torch.tensor(cs)

# --- 2. A tiny CNN composed with torch.nn. conv2 is the LAST conv -> feature maps A^k. ---
class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, 3, padding=1)
        self.conv2 = nn.Conv2d(8, 8, 3, padding=1)     # last conv layer
        self.fc    = nn.Linear(8, 2)
    def features(self, x):
        x = torch.relu(self.conv1(x))
        return torch.relu(self.conv2(x))               # (n,8,8,8) = feature maps A^k
    def forward(self, x):
        A = self.features(x)
        return self.fc(A.mean(dim=(2,3))), A           # GAP -> logits, plus A

net = TinyCNN(); opt = torch.optim.Adam(net.parameters(), lr=0.01)
lossf = nn.CrossEntropyLoss()
for step in range(60):
    X, y = batch(64); opt.zero_grad()
    logits, _ = net(X); loss = lossf(logits, y)
    loss.backward(); opt.step()
Xt, yt = batch(400)
with torch.no_grad():
    acc = (net(Xt)[0].argmax(1) == yt).float().mean().item()
print("\ntrain loss = %.4f, held-out accuracy = %.3f" % (loss.item(), acc))
# train loss = 0.0520, held-out accuracy = 1.000


# --- 3. Grad-CAM, by hand (Eqn 1 + Eqn 2). ---
def gradcam(img, target_class):
    A = net.features(img)            # (1,8,8,8) last-conv feature maps
    A.retain_grad()                  # keep grad on this intermediate tensor
    logits = net.fc(A.mean(dim=(2,3)))
    net.zero_grad()
    logits[0, target_class].backward()        # backprop the LOGIT y^c (not softmax prob)
    grads = A.grad[0]                          # dy^c / dA^k_ij   shape (8,8,8)
    alpha = grads.mean(dim=(1,2))             # Eqn 1: GAP over space -> weight per map
    cam = torch.relu((alpha.view(-1,1,1) * A[0]).sum(0))   # Eqn 2: ReLU(sum_k alpha_k A^k)
    cam = cam.detach().numpy()
    return cam / cam.max() if cam.max() > 0 else cam

# Build one canonical CLASS-0 image (blob centered at top-left cell (1,1)).
np.random.seed(7)
img0 = np.random.uniform(0, 0.1, (H,H)).astype(np.float32)
for dy in (-1,0,1):
    for dx in (-1,0,1): img0[1+dy, 1+dx] += 1.0
img = torch.tensor(img0).unsqueeze(0).unsqueeze(0)

cam0 = gradcam(img, 0)    # "why class 0?"  -> should light TOP-LEFT (the blob)
cam1 = gradcam(img, 1)    # "why class 1?"  -> should move AWAY from the blob
np.set_printoptions(precision=2, suppress=True)
print("\nGrad-CAM for target class 0 (correct class):"); print(np.round(cam0,2))
print("  top-left 3x3 mean = %.3f, bottom-right 3x3 mean = %.3f"
      % (cam0[:3,:3].mean(), cam0[5:,5:].mean()))
print("\nGrad-CAM for target class 1 (the other class):"); print(np.round(cam1,2))
print("  top-left 3x3 mean = %.3f, bottom-right 3x3 mean = %.3f"
      % (cam1[:3,:3].mean(), cam1[5:,5:].mean()))
# class 0 map:  top-left mean ~0.55, bottom-right mean ~0.13  (highlights the blob)
# class 1 map:  top-left mean ~0.22, bottom-right mean ~0.01  (highlight moves off the blob)
# Our small run, not the paper's reported numbers. Exact values vary by seed/hardware.

## Visualize the data & results

_On ONE image that has a bright blob in the top-left (true class 0), what does the Grad-CAM heatmap highlight when we ask 'why class 0?' versus 'why class 1?' — does the hot region move when we change the target class?_

In [ ]:
import torch, torch.nn as nn, numpy as np
torch.manual_seed(0); np.random.seed(0)
H = 8
def make_image(cls):
    img = np.random.uniform(0, 0.1, (H,H)).astype(np.float32)
    cy, cx = (np.random.randint(0,3), np.random.randint(0,3)) if cls==0 \
             else (np.random.randint(5,8), np.random.randint(5,8))
    for dy in (-1,0,1):
        for dx in (-1,0,1):
            y,x = cy+dy, cx+dx
            if 0<=y<H and 0<=x<H: img[y,x]+=1.0
    return img
def batch(n):
    cs = np.random.randint(0,2,n)
    return torch.tensor(np.stack([make_image(c) for c in cs])).unsqueeze(1), torch.tensor(cs)

class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1,8,3,padding=1)
        self.conv2 = nn.Conv2d(8,8,3,padding=1)   # last conv -> feature maps A^k
        self.fc    = nn.Linear(8,2)
    def features(self, x):
        x = torch.relu(self.conv1(x)); return torch.relu(self.conv2(x))
    def forward(self, x):
        A = self.features(x); return self.fc(A.mean(dim=(2,3))), A

net = TinyCNN(); opt = torch.optim.Adam(net.parameters(), lr=0.01); lf = nn.CrossEntropyLoss()
for _ in range(60):
    X,y = batch(64); opt.zero_grad(); lo,_ = net(X); l = lf(lo,y); l.backward(); opt.step()
Xt,yt = batch(400)
with torch.no_grad(): acc = (net(Xt)[0].argmax(1)==yt).float().mean().item()
print("held-out accuracy:", round(acc,3))

def gradcam(img, c):
    A = net.features(img); A.retain_grad()
    net.zero_grad(); net.fc(A.mean(dim=(2,3)))[0,c].backward()   # backprop logit y^c
    alpha = A.grad[0].mean(dim=(1,2))                            # Eqn 1: GAP of gradients
    cam = torch.relu((alpha.view(-1,1,1)*A[0]).sum(0)).detach().numpy()  # Eqn 2
    return cam/cam.max() if cam.max()>0 else cam

np.random.seed(7)
img0 = np.random.uniform(0,0.1,(H,H)).astype(np.float32)
for dy in (-1,0,1):
    for dx in (-1,0,1): img0[1+dy,1+dx]+=1.0
img = torch.tensor(img0).unsqueeze(0).unsqueeze(0)
cam0, cam1 = gradcam(img,0), gradcam(img,1)
np.set_printoptions(precision=2, suppress=True)
print("class-0 map:\n", np.round(cam0,2))
print("class-1 map:\n", np.round(cam1,2))
print("blob (top-left 3x3) mean -- class 0: %.3f, class 1: %.3f"
      % (cam0[:3,:3].mean(), cam1[:3,:3].mean()))
# held-out accuracy: 1.0
# blob (top-left 3x3) mean -- class 0: 0.550, class 1: 0.215
# Same image, different target class -> the highlight moves. Our small run, not the paper's number.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ReLU ablation. You have a working Grad-CAM. Remove the ReLU from Equation 2 &mdash; use the
            raw weighted sum $\sum_k \alpha_k^c A^k$ &mdash; everything else identical, and recompute the map for
            the correct class. What changes in the heatmap, and why does the paper insist on the ReLU?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Locate Equation 2 in code: cam = relu((alpha[:,None,None] * A).sum(0)). Drop the relu so negative cells survive. — _The ReLU is the only thing clipping negatives; removing it lets cells that push the class score down appear in the map._
- Recompute and inspect: regions that argue against class $c$ now show as negative values; after you normalize for display, they steal contrast from the true object. — _Without clipping, "evidence against the class" is mixed into a map that is supposed to mean "evidence for the class," so the highlight gets noisier and less localized._
- Conclude: the ReLU keeps only positive influence, matching the paper's stated reason. — _&sect;3: "We apply a ReLU ... because we are only interested in the features that have a positive influence on the class of interest."_

**Answer:** Dropping the ReLU lets cells with negative influence on class $c$ into the map. The
                 heatmap stops meaning "regions that support class $c$" and becomes a signed blend of for-and-against
                 evidence; after normalization the true object's highlight is diluted by anti-class pixels. The
                 paper keeps the ReLU exactly to avoid this &mdash; it is "only interested in the features that have
                 a positive influence on the class of interest" (&sect;3). So the ReLU is what makes the map a clean
                 positive explanation.

</details>

**Problem 2.** Class-discriminativeness. On one image that truly has a top-left blob (class 0), you compute
            two Grad-CAM maps from the same forward features: one for $c=0$ and one for $c=1$. Why do the
            two maps differ, even though $A^k$ is identical for both?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note what is shared and what is not: the feature maps $A^k$ come from one forward pass and are the same for both targets; only the class score $y^c$ you back-propagate changes. — _Equation 2 weights the same maps, but with weights $\alpha_k^c$ that depend on $c$._
- Trace Equation 1 for each class: $\alpha_k^0$ pools $\partial y^0/\partial A^k$, $\alpha_k^1$ pools $\partial y^1/\partial A^k$. Different target logit &rarr; different gradient &rarr; different weights. — _The classifier weights for class 0 and class 1 differ, so each class is sensitive to a different mix of feature maps._
- Assemble: the class-0 weights emphasize top-left-firing maps (the blob is class-0 evidence), so that map lights the top-left; the class-1 weights emphasize bottom-right-firing maps, so its map shifts away from the blob. — _Same patterns, re-weighted per class, produce different localizations &mdash; the definition of class-discriminative._

**Answer:** Because only the weights $\alpha_k^c$ depend on the class, and they come from
                 differentiating a different logit (Equation 1). The feature maps $A^k$ are shared, but
                 class 0 and class 1 are sensitive to different maps, so global-average-pooling their gradients
                 gives different weights. The weighted sum (Equation 2) then highlights different regions. In our
                 run, the "why class 0?" map lights the top-left blob (mean &asymp; 0.55 there) while the "why class
                 1?" map abandons it (mean &asymp; 0.215, and the blob's own pixels go to 0). Same image, different
                 question, different heatmap.

</details>

**Problem 3.** In the worked example a single $2\times2$ map had gradient $\big[\begin{smallmatrix}1&3\\-2&0\end{smallmatrix}\big]$,
            giving $\alpha_1^c = 0.5$ and heatmap $\big[\begin{smallmatrix}1.0&0\\0.5&2.0\end{smallmatrix}\big]$.
            Now suppose the gradient were instead $\big[\begin{smallmatrix}-1&-3\\-2&0\end{smallmatrix}\big]$
            (the same map $A^1=\big[\begin{smallmatrix}2&0\\1&4\end{smallmatrix}\big]$). Recompute $\alpha_1^c$
            and the heatmap. What does the result tell you about this map's role for class $c$?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Equation 1: sum the gradient cells $-1 + (-3) + (-2) + 0 = -6$; divide by $Z=4$: $\alpha_1^c = -1.5$. — _Now the map's gradients are mostly negative, so its pooled importance is negative._
- Equation 2 (one map): weighted map $= -1.5 \cdot \big[\begin{smallmatrix}2&0\\1&4\end{smallmatrix}\big] = \big[\begin{smallmatrix}-3.0&0\\-1.5&-6.0\end{smallmatrix}\big]$. Apply ReLU: every cell is $\le 0$, so it all clips to $\big[\begin{smallmatrix}0&0\\0&0\end{smallmatrix}\big]$. — _A negatively-weighted map produces a non-positive contribution; the ReLU removes it entirely._
- Interpret: this map argues against class $c$, so it contributes nothing to the class-$c$ explanation. — _Grad-CAM keeps only positive influence; an anti-class map is exactly what the ReLU is there to suppress._

**Answer:** $\alpha_1^c = -6/4 = -1.5$. The weighted map is all $\le 0$, so after the ReLU the heatmap is
                 all zeros. A negative $\alpha_k^c$ means the map pushes the class-$c$ score down &mdash; it
                 is evidence against class $c$ &mdash; and the ReLU drops its contribution. Contrast with the
                 original positive $\alpha_1^c = 0.5$, where the map survived and highlighted the bottom-right
                 cell. This is exactly why Grad-CAM averages the gradient (so anti-class maps get negative
                 weight) and then ReLUs (so they vanish from the map).

</details>